# Kumo Engine `result.summary()` demo

This notebook runs Kumo Engine from the `prototype/result-summary` branch on the Brev instance `kumo-engine-experimental`.
Steps 1 to 3 install it and check that the notebook really uses it.
Step 4 onward fits one small, seeded synthetic example per task type and shows `result.summary()`, binary classification first.

The scores illustrate the report, not model quality, and change slightly between runs.
Each fit takes under a minute on the instance's L4 GPU; the ten-experiment binary fit takes a few minutes.

## Step 1. Install Kumo Engine from this branch

The checkout `/home/ubuntu/kumo-engine-fork` is on `prototype/result-summary`.
An editable install makes its virtual environment import Engine straight from this source tree, so `result.summary()` is the branch's code.
Its dependencies, including Kumo Core, are already installed in that environment, so only Engine is installed here.

In [1]:
%%bash
set -eo pipefail
cd /home/ubuntu/kumo-engine-fork
git branch --show-current
git log --oneline -1
~/.local/bin/uv pip install --python .venv/bin/python --no-deps -e packages/kumo-engine 2>&1 | tail -3

prototype/result-summary


0872f59 Show every recorded metric's curve with hover values in result.summary()


Uninstalled 1 package in 0.46ms
Installed 1 package in 0.75m

s
 ~ kumo-engine==1.0.0.dev0 (from file:///home/ubuntu/kumo-engine-for

k/packages/kumo-engine)


## Step 2. Select the kernel

This registers the environment as the `result-summary` Jupyter kernel.
The notebook already selects it; otherwise choose **Kernel > Change Kernel > Kumo Engine result.summary prototype**.
The check below fails if another kernel is running.

In [2]:
%%bash
set -eo pipefail
/home/ubuntu/kumo-engine-fork/.venv/bin/python -m ipykernel install --prefix /home/ubuntu/kumo-engine-fork/.venv --name result-summary --display-name "Kumo Engine result.summary prototype"

Installed kernelspec result-summary in /home/ubuntu/kumo-engine-fork/.venv/share/jupyter/kernels/res

ult-summary


In [3]:
import sys
from pathlib import Path

REPO = Path('/home/ubuntu/kumo-engine-fork')
assert Path(sys.prefix) == REPO / ".venv", sys.prefix
print("Kernel environment:", sys.prefix)

Kernel environment: /home/ubuntu/kumo-engine-fork/.venv


## Step 3. Check that Engine is this branch's code

In [4]:
import inspect
import subprocess
import warnings

# tqdm warns that ipywidgets is absent; fits here show no progress bars.
warnings.filterwarnings("ignore", message="IProgress not found")
import kumo.engine
from kumo.engine.trainer.trainer import TrainingResult


def git(*args):
    return subprocess.run(["git", "-C", REPO, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


source = Path(inspect.getsourcefile(TrainingResult.summary))
assert Path(kumo.engine.__file__).is_relative_to(REPO / "packages/kumo-engine")
assert source.is_relative_to(REPO / "packages/kumo-engine")
assert git("branch", "--show-current") == 'prototype/result-summary'
print("Engine imported from:", Path(kumo.engine.__file__).parent)
print("summary() defined in:", source)
print("Branch and commit:   ", git("branch", "--show-current"), git("log", "--oneline", "-1"))
print("Uncommitted Engine changes:", git("status", "--porcelain", "--", "packages/kumo-engine") or "none")

Engine imported from: /home/ubuntu/kumo-engine-fork/packages/kumo-engine/kumo/engine
summary() defined in: /home/ubuntu/kumo-engine-fork/packages/kumo-engine/kumo/engine/trainer/trainer.py
Branch and commit:    prototype/result-summary 0872f59 Show every recorded metric's curve with hover values in result.summary()
Uncommitted Engine changes: none


## Step 4. Example data

`example(task)` builds a small seeded synthetic graph, training table and trainer for a task, using the generators in `artifacts/result-summary/validation_harness.py`.
Fits run inside `quiet()`, which hides their ingest and training logs; errors still raise.
Models are written under `~/result-summary-demo-runs`, outside the checkout.

In [5]:
import contextlib, importlib.util, logging, os, tempfile, warnings

warnings.filterwarnings("ignore")
spec = importlib.util.spec_from_file_location(
    "validation_harness", REPO / "artifacts/result-summary/validation_harness.py")
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)
RUNS = Path.home() / "result-summary-demo-runs"
RUNS.mkdir(exist_ok=True)
OUT = Path(tempfile.mkdtemp(dir=RUNS))


@contextlib.contextmanager
def quiet():
    """Hide the ingest and training logs of a fit; errors still raise."""
    sys.stdout.flush(), sys.stderr.flush()
    saved = os.dup(1), os.dup(2)
    logging.disable(logging.CRITICAL)
    try:
        with open(os.devnull, "w") as null, contextlib.redirect_stdout(null), \
                contextlib.redirect_stderr(null):
            os.dup2(null.fileno(), 1), os.dup2(null.fileno(), 2)
            yield
    finally:
        logging.disable(logging.NOTSET)
        for fd, copy in zip((1, 2), saved):
            os.dup2(copy, fd), os.close(copy)


def example(task, scenario="baseline"):
    """A trainer, training table and snapshot for one small task."""
    with quiet():
        ctx = h.prepare(REPO, OUT / task, task, scenario, "result-summary demo")
    return ctx["trainer"], ctx["table"], ctx["snapshot"]

## Step 5. Binary classification

Ten experiments. In the Training tab, the experiment dropdown starts on the best one, marked ★, which is the published model; the metric dropdown shows any metric that experiment recorded per epoch, and hovering a point shows its value. The Evaluation tab has the threshold slider and confusion matrix.

In [6]:
trainer, training_table, snapshot = example('binary', 'many_trials')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [7]:
result.summary()

Evaluation metrics
Metric | Training | Validation | Holdout
AUROC * | 0.690 | 0.939 | 0.851
Accuracy | 0.500 | 0.451 | 0.634
AP | 0.636 | 0.953 | 0.803
AUPRC | 0.631 | 0.952 | 0.797
Brier score | - | - | 0.264
F1 | 0.000 | 0.000 | 0.000
Log loss | - | - | 0.721
Precision | 0.000 | 0.000 | 0.000
Recall | 0.000 | 0.000 | 0.000
Learning curves: Experiment 10 ★
  train loss: 0.754, 0.700, 0.699, 0.699
  val loss: 0.686, 0.688, 0.688, 0.731
  train auroc: 0.640, 0.584, 0.690, 0.633
  val auroc: 0.899, 0.936, 0.939, 0.073
No per-epoch history: Brier score, Log loss

## Step 6. Multiclass classification

Per-class metrics.

In [8]:
trainer, training_table, snapshot = example('multiclass')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [9]:
result.summary()

Evaluation metrics
Metric | Training | Validation | Holdout
Accuracy * | 0.385 | 0.706 | 0.561
F1 | 0.313 | 0.467 | 0.392
F1 (macro) | - | - | 0.392
F1 (micro) | - | - | 0.561
F1 (weighted) | - | - | 0.444
Log loss | - | - | 0.992
Precision | 0.336 | 0.483 | 0.411
Recall | 0.354 | 0.521 | 0.498
Learning curves: Experiment 1 ★
  train loss: 1.484, 1.284, 1.186, 1.164
  val loss: 1.362, 1.082, 1.024, 1.003
  train accuracy: 0.291, 0.385, 0.446, 0.459
  val accuracy: 0.137, 0.706, 0.686, 0.706
No per-epoch history: F1 (macro), F1 (micro), F1 (weighted), Log loss

## Step 7. Multilabel classification

Macro and micro metrics, then per-label metrics.

In [10]:
trainer, training_table, snapshot = example('multilabel')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [11]:
result.summary()

Metric,Training,Validation,Holdout
AP (macro) *,0.638,0.756,0.649
AP (micro),0.667,0.700,0.693
AUPRC (macro),0.631,0.747,0.631
AUPRC (micro),0.666,0.699,0.691
AUROC (macro),0.622,0.730,0.676
AUROC (micro),0.663,0.665,0.677
F1 (macro),-,-,0.625
F1 (micro),-,-,0.654
Precision (micro),-,-,0.607
Recall (micro),-,-,0.710


## Step 8. Multilabel ranking

Ranking metrics at each cutoff K.

In [12]:
trainer, training_table, snapshot = example('multilabel_ranking')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [13]:
result.summary()

Metric,K,Training,Validation,Holdout
Recall,1,0.255,0.145,0.193
,5 *,0.895,0.874,0.897
,10,1.000,1.000,1.000
F1,1,0.357,0.222,0.272
,5,0.652,0.646,0.632
,10,0.460,0.463,0.448
MAP,1,0.716,0.529,0.537
,5,0.689,0.651,0.666
,10,0.744,0.704,0.731
NDCG,1,0.716,0.529,0.537


## Step 9. Regression

Error metrics.

In [14]:
trainer, training_table, snapshot = example('regression')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [15]:
result.summary()

Metric,Training,Validation,Holdout
MAE *,1.413,1.383,1.367
Bias,-,-,-0.226
MAPE,-,-,0.558
MSE,3.306,3.061,2.830
R²,-,-,-0.018
RMSE,1.818,1.750,1.682
SMAPE,0.561,0.529,0.530


## Step 10. Forecasting

Error metrics and error by forecast horizon.

In [16]:
trainer, training_table, snapshot = example('forecasting')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [17]:
result.summary()

Evaluation metrics
Metric | Training | Validation | Holdout
MAE * | 1.438 | 1.350 | 1.356
Bias | - | - | -0.110
MAPE | - | - | 0.577
MSE | 3.307 | 2.882 | 2.819
R² | - | - | -0.004
RMSE | 1.818 | 1.698 | 1.679
SMAPE | 0.554 | 0.520 | 0.515
Learning curves: Experiment 1 ★
  train loss: 2.679, 1.875, 1.563, 1.438
  val loss: 1.926, 1.529, 1.410, 1.350
  train mae: 2.679, 1.875, 1.563, 1.438
  val mae: 1.926, 1.529, 1.410, 1.350
No per-epoch history: Bias, MAPE, R²

## Step 11. Static link prediction

Ranking metrics at each cutoff K.

In [18]:
trainer, training_table, snapshot = example('static_link')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [19]:
result.summary()

Metric,K,Training,Validation,Holdout
Recall,1,0.040,0.123,0.232
,5 *,0.208,0.558,0.882
,10,0.425,0.942,1.000
F1,1,0.075,0.188,0.357
,5,0.307,0.486,0.809
,10,0.497,0.554,0.629
MAP,1,0.587,0.496,0.928
,5,0.462,0.417,0.925
,10,0.432,0.570,0.952
NDCG,1,0.587,0.496,0.928


## Step 12. Temporal link prediction

Ranking metrics at each cutoff K.

In [20]:
trainer, training_table, snapshot = example('temporal_link')
with quiet():
    result = trainer.fit(training_table, snapshot=snapshot)

In [21]:
result.summary()

Metric,K,Training,Validation,Holdout
Recall,1,0.050,0.041,0.040
,5,0.213,0.210,0.211
,10 *,0.424,0.435,0.412
F1,1,0.069,0.057,0.056
,5,0.143,0.146,0.145
,10,0.179,0.186,0.177
LOCALITY (score),,-,-,0.499
MAP,1,0.141,0.116,0.114
,5,0.113,0.108,0.103
,10,0.154,0.152,0.142
